In [1]:
import importlib.util
import json
import warnings
from pathlib import Path
from typing import Dict, Optional, Tuple

import torch
import torch.nn.functional as F
from safetensors.torch import save_file
from torchvision.io import read_video


DATASET_ROOT = Path("/home/azureuser/datasets/navier")
WAN_REPO_ROOT = Path("/home/azureuser/physics/navier/Wan2.2")
WAN_CHECKPOINT_DIR = Path("/home/azureuser/Wan2.2-TI2V-5B")
WAN_TI2V_5B_VAE_CHECKPOINT = "Wan2.2_VAE.pth"
WAN_TI2V_5B_VAE_STRIDE = (4, 16, 16)


def _load_wan2_2_vae_class(wan_repo_root: str | Path = WAN_REPO_ROOT):
    """Load Wan2.2's VAE class without importing wan/__init__.py extras."""
    module_path = Path(wan_repo_root) / "wan" / "modules" / "vae2_2.py"
    if not module_path.exists():
        raise FileNotFoundError(f"Wan2.2 VAE module not found: {module_path}")

    spec = importlib.util.spec_from_file_location("wan_vae2_2_local", module_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not create import spec for {module_path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module.Wan2_2_VAE


def load_wan_ti2v_5b_vae(
    checkpoint_dir: str | Path = WAN_CHECKPOINT_DIR,
    wan_repo_root: str | Path = WAN_REPO_ROOT,
    device: Optional[str | torch.device] = None,
    dtype: torch.dtype = torch.float32,
):
    """Load the Wan2.2 TI2V-5B VAE from /home/azureuser/Wan2.2-TI2V-5B."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") if device is None else torch.device(device)
    checkpoint_path = Path(checkpoint_dir) / WAN_TI2V_5B_VAE_CHECKPOINT
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"VAE checkpoint not found: {checkpoint_path}")

    Wan2_2_VAE = _load_wan2_2_vae_class(wan_repo_root)
    vae = Wan2_2_VAE(
        vae_pth=str(checkpoint_path),
        dtype=dtype,
        device=device,
    )
    return vae


def load_video_for_wan_vae(
    video_path: str | Path,
    device: Optional[str | torch.device] = None,
    expected_size: Tuple[int, int] = (256, 256),
    resize: bool = False,
) -> Tuple[torch.Tensor, Dict]:
    """Read an mp4 and return Wan-normalized video shaped (C, T, H, W)."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") if device is None else torch.device(device)
    video_path = Path(video_path)
    if not video_path.exists():
        raise FileNotFoundError(video_path)

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UserWarning, module="torchvision")
        frames, _, info = read_video(str(video_path), pts_unit="sec", output_format="TCHW")
    if frames.numel() == 0:
        raise ValueError(f"No video frames decoded from {video_path}")

    if frames.shape[1] < 3:
        frames = frames.repeat(1, 3, 1, 1)
    frames = frames[:, :3]
    frames = frames.float().div_(255.0) if frames.dtype == torch.uint8 else frames.float().clamp_(0.0, 1.0)

    target_h, target_w = expected_size
    if tuple(frames.shape[-2:]) != (target_h, target_w):
        if not resize:
            raise ValueError(
                f"Expected {expected_size} video frames, got {tuple(frames.shape[-2:])} for {video_path}"
            )
        frames = F.interpolate(frames, size=expected_size, mode="bicubic", align_corners=False).clamp_(0.0, 1.0)

    # Wan preprocessing uses TF.to_tensor(...).sub_(0.5).div_(0.5).
    video = frames.mul_(2.0).sub_(1.0).permute(1, 0, 2, 3).contiguous().to(device)
    return video, dict(info)


@torch.no_grad()
def encode_video_to_wan_latents(
    video_path: str | Path,
    vae=None,
    checkpoint_dir: str | Path = WAN_CHECKPOINT_DIR,
    wan_repo_root: str | Path = WAN_REPO_ROOT,
    device: Optional[str | torch.device] = None,
    vae_dtype: torch.dtype = torch.float32,
    save_dtype: torch.dtype = torch.float16,
    output_path: Optional[str | Path] = None,
    overwrite: bool = False,
    expected_size: Tuple[int, int] = (256, 256),
    resize: bool = False,
) -> Dict[str, str | int | float | Tuple[int, ...]]:
    """Encode one dataset mp4 and save latents.safetensors beside it."""
    video_path = Path(video_path)
    output_path = video_path.with_name("latents.safetensors") if output_path is None else Path(output_path)
    if output_path.exists() and not overwrite:
        return {"status": "skipped", "reason": "exists", "video": str(video_path), "latents": str(output_path)}

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") if device is None else torch.device(device)
    if vae is None:
        vae = load_wan_ti2v_5b_vae(checkpoint_dir, wan_repo_root, device=device, dtype=vae_dtype)

    video, video_info = load_video_for_wan_vae(video_path, device=device, expected_size=expected_size, resize=resize)
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=FutureWarning, message=".*torch.cuda.amp.autocast.*")
        latents = vae.encode([video])[0].detach().to("cpu", dtype=save_dtype).contiguous()

    metadata = {
        "source_video": video_path.name,
        "vae": "Wan2.2 TI2V-5B VAE",
        "vae_checkpoint": str(Path(checkpoint_dir) / WAN_TI2V_5B_VAE_CHECKPOINT),
        "vae_stride": json.dumps(WAN_TI2V_5B_VAE_STRIDE),
        "normalization": "uint8 [0,255] -> [0,1] -> [-1,1]",
        "input_shape_cthw": json.dumps(tuple(video.shape)),
        "latent_shape_cthw": json.dumps(tuple(latents.shape)),
        "latent_dtype": str(latents.dtype),
        "video_info": json.dumps(video_info, sort_keys=True),
    }
    save_file({"latents": latents}, str(output_path), metadata=metadata)
    return {
        "status": "ok",
        "video": str(video_path),
        "latents": str(output_path),
        "frames": int(video.shape[1]),
        "latent_shape": tuple(latents.shape),
    }


def add_wan_vae_latents_to_dataset(
    dataset_root: str | Path = DATASET_ROOT,
    checkpoint_dir: str | Path = WAN_CHECKPOINT_DIR,
    wan_repo_root: str | Path = WAN_REPO_ROOT,
    device: Optional[str | torch.device] = None,
    vae_dtype: torch.dtype = torch.float32,
    save_dtype: torch.dtype = torch.float16,
    overwrite: bool = False,
    limit: Optional[int] = None,
    show_progress: bool = True,
    continue_on_error: bool = True,
    expected_size: Tuple[int, int] = (256, 256),
    resize: bool = False,
) -> list[Dict]:
    """Scan dataset_root/*/video.mp4 and add latents.safetensors to each subfolder."""
    dataset_root = Path(dataset_root)
    all_video_paths = sorted(dataset_root.glob("*/video.mp4"))
    if limit is not None:
        all_video_paths = all_video_paths[: int(limit)]
    if not all_video_paths:
        return []

    records = []
    video_paths = []
    for video_path in all_video_paths:
        latents_path = video_path.with_name("latents.safetensors")
        if latents_path.exists() and not overwrite:
            records.append({"status": "skipped", "reason": "exists", "video": str(video_path), "latents": str(latents_path)})
        else:
            video_paths.append(video_path)
    if not video_paths:
        return records

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu") if device is None else torch.device(device)
    vae = load_wan_ti2v_5b_vae(checkpoint_dir, wan_repo_root, device=device, dtype=vae_dtype)

    iterator = video_paths
    if show_progress:
        try:
            from tqdm.auto import tqdm
            iterator = tqdm(video_paths, desc="Encoding Wan VAE latents")
        except Exception:
            pass

    for video_path in iterator:
        try:
            record = encode_video_to_wan_latents(
                video_path,
                vae=vae,
                checkpoint_dir=checkpoint_dir,
                wan_repo_root=wan_repo_root,
                device=device,
                vae_dtype=vae_dtype,
                save_dtype=save_dtype,
                overwrite=overwrite,
                expected_size=expected_size,
                resize=resize,
            )
        except Exception as exc:
            if not continue_on_error:
                raise
            record = {"status": "error", "video": str(video_path), "error": repr(exc)}
        records.append(record)
    return records


# Example full pass:
records = add_wan_vae_latents_to_dataset()

# Example smoke test:
# records = add_wan_vae_latents_to_dataset(limit=1)


/opt/miniforge/envs/new/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Encoding Wan VAE latents: 100%|██████████| 1000/1000 [06:17<00:00,  2.65it/s]
